# Phase 2: Multi-Representation Normalization Engine
## Amazon ML Challenge 2026: Business Entity Resolution

### Objective:
Build a robust, multi-representation Normalization Engine that standardizes business names and addresses across **US, India, and France** without destroying discriminative identity information.

### Key Capabilities Implemented:
1. **Unicode Script & Diacritic Preservation**: `NFKC` Unicode standardization preserving Indian scripts (Tamil, Hindi) and French accents (`é`, `è`, `ç`, `ô`).
2. **Multi-Region Legal Suffix Standardization**: US (`Inc`, `Corp`, `LLC`), India (`Pvt Ltd`, `LLP`), and France (`SARL`, `SAS`, `SA`, `EURL`).
3. **Core Name Extraction**: Stripping legal suffixes to allow direct matching when suffixes are omitted in noisy sources.
4. **Web Domain / URL Stripping**: Standardizing domain-formatted names (`company.com` $\rightarrow$ `company`).
5. **Token Sorting Invariance**: Neutralizing word-order transpositions (`Reliable Scientific Power` $\leftrightarrow$ `Power Reliable Scientific`).
6. **Structural Address Anchors**: Extracting 5–6 digit PINs, numeric house/street tokens, road markers, and missing address flags.

---

In [1]:
import os
import sys
import pandas as pd

# Change working directory to project root if running from notebooks directory
if os.path.basename(os.getcwd()) == 'notebooks':
    os.chdir('..')

sys.path.insert(0, os.getcwd())
print("Working directory:", os.getcwd())

from src.business_entity_resolution.normalization import (
    normalize_name,
    normalize_address,
    clean_unicode_text,
    strip_web_domains,
    process_dataframe
)
from src.business_entity_resolution.io import load_train_data

Working directory: D:\ML Challenge 2026\amazon


## 1. Multi-Script & International Unicode Preservation
Demonstrating that Tamil, Hindi (Devanagari), and French accented entities are standardized without lossy ASCII corruption.

In [2]:
multi_script_samples = [
    ("Tamil Script", "ராஜ் இன்வெஸ்ட்மெண்ட்ஸ் எல்எல்பி", "6(29), C.I.T. Colony, Chennai"),
    ("Hindi Script", "एसएस फूड प्राइवेट लिमिटेड", "Af-684, Nandgram Near Mother India"),
    ("French Diacritics", "Dréxkor S.A.R.L.", "85 Rue de la République, 75001 Paris"),
    ("English Reference", "Raj Investments LLP", "6(29), C.I.T. Colony, 2nd Main Rd")
]

script_results = []
for lang, name, addr in multi_script_samples:
    n_res = normalize_name(name)
    a_res = normalize_address(addr)
    script_results.append({
        "Category": lang,
        "Raw Name": name,
        "Clean Norm": n_res["clean_norm"],
        "Core Name": n_res["core_name"],
        "Suffix Norm": n_res["suffix_norm"],
        "Address PIN": a_res["pin"],
        "Address Numbers": a_res["numbers"]
    })

display(pd.DataFrame(script_results))

,Category,Raw Name,Clean Norm,Core Name,Suffix Norm,Address PIN,Address Numbers
0,Tamil Script,ராஜ் இன்வெஸ்ட்மெண்ட்ஸ் எல்எல்பி,ர ஜ இன வ ஸ ட ம ண ட ஸ எல எல ப,ர ஜ இன வ ஸ ட ம ண ட ஸ எல எல ப,ர ஜ இன வ ஸ ட ம ண ட ஸ எல எல ப,,"[6, 29]"
1,Hindi Script,एसएस फूड प्राइवेट लिमिटेड,एसएस फ ड प र इव ट ल म ट ड,एसएस फ ड प र इव ट ल म ट ड,एसएस फ ड प र इव ट ल म ट ड,,[684]
2,French Diacritics,Dréxkor S.A.R.L.,dréxkor s a r l,dréxkor,dréxkor societe a responsabilite limitee,75001,"[85, 75001]"
3,English Reference,Raj Investments LLP,raj investments llp,raj investments,raj investments limited liability partnership,,"[6, 29]"


## 2. Multi-Region Legal Suffix Standardization & Core Stripping
Evaluating legal suffix expansions (`suffix_norm`) and core business name representations (`core_name`) across US, India, and France.

In [3]:
suffix_samples = [
    ("US Entity (Inc)", "Acme Technologies Inc."),
    ("US Entity (Corp)", "Apex Global Corp."),
    ("US Entity (LLC)", "Dahlia Scientific Power LLC"),
    ("India Entity (Pvt Ltd)", "Ss Food Private Limited"),
    ("India Entity (Abbrev)", "Ss Food Pvt. Ltd."),
    ("India Entity (LLP)", "Raj Investments LLP"),
    ("France Entity (SARL)", "Boulangerie Moderne S.A.R.L."),
    ("France Entity (SAS)", "Transport Express SASU")
]

suffix_results = []
for region, name in suffix_samples:
    res = normalize_name(name)
    suffix_results.append({
        "Region": region,
        "Raw Name": name,
        "Clean Norm": res["clean_norm"],
        "Suffix Norm (Expanded)": res["suffix_norm"],
        "Core Name (Stripped)": res["core_name"],
        "Informative Tokens": res["informative_tokens"]
    })

display(pd.DataFrame(suffix_results))

,Region,Raw Name,Clean Norm,Suffix Norm (Expanded),Core Name (Stripped),Informative Tokens
0,US Entity (Inc),Acme Technologies Inc.,acme technologies inc,acme technologies incorporated,acme technologies,[acme]
1,US Entity (Corp),Apex Global Corp.,apex global corp,apex global corporation,apex global,[apex]
2,US Entity (LLC),Dahlia Scientific Power LLC,dahlia scientific power llc,dahlia scientific power limited liability company,dahlia scientific power,"[dahlia, scientific, power]"
3,India Entity (Pvt Ltd),Ss Food Private Limited,ss food private limited,ss food private limited,ss food,"[ss, food]"
4,India Entity (Abbrev),Ss Food Pvt. Ltd.,ss food pvt ltd,ss food private limited,ss food,"[ss, food]"
5,India Entity (LLP),Raj Investments LLP,raj investments llp,raj investments limited liability partnership,raj investments,"[raj, investments]"
6,France Entity (SARL),Boulangerie Moderne S.A.R.L.,boulangerie moderne s a r l,boulangerie moderne societe a responsabilite l...,boulangerie moderne,"[boulangerie, moderne]"
7,France Entity (SAS),Transport Express SASU,transport express sasu,transport express societe par actions simplifi...,transport express,"[transport, express]"


## 3. Web Domain Stripping & Word-Order Sorting
1. **Domain Stripping**: Aligns URL representations with standard names.
2. **Token Sorting**: Solves word-order transpositions.

In [4]:
# Domain tests
domain_samples = [
    "maurewilliamscolombier.com",
    "https://www.globaltech.in",
    "www.apex-logistics.org",
    "boulangerie-paris.fr"
]
print("--- Web Domain Stripping ---")
for d in domain_samples:
    res = normalize_name(d)
    print(f"Raw: {d:<30} -> Clean: {res['clean_norm']}")

# Token Sorting tests
print("\n--- Token Sorting Invariance ---")
t1 = normalize_name("Reliable Scientific Power LLC")
t2 = normalize_name("Power Reliable Scientific Inc")
print(f"Name 1 Sorted Tokens: '{t1['sorted_tokens']}'")
print(f"Name 2 Sorted Tokens: '{t2['sorted_tokens']}'")
print(f"Match Invariance Achieved: {t1['sorted_tokens'] == t2['sorted_tokens']}")

--- Web Domain Stripping ---
Raw: maurewilliamscolombier.com     -> Clean: maurewilliamscolombier
Raw: https://www.globaltech.in      -> Clean: globaltech
Raw: www.apex-logistics.org         -> Clean: apex logistics
Raw: boulangerie-paris.fr           -> Clean: boulangerie paris

--- Token Sorting Invariance ---
Name 1 Sorted Tokens: 'power reliable scientific'
Name 2 Sorted Tokens: 'power reliable scientific'
Match Invariance Achieved: True


## 4. Address Structural Anchors & Missing Address Handling
Extracting invariant structural components (PIN codes, house/building numbers, road terms) and flagging missing records.

In [5]:
address_samples = [
    "85 Wayne Avenue, Ticonderoga, NY 12883",
    "Wayne Ave, Ticonderoga Townshiip, NY",
    "6(29), C.I.T. Colony, 2nd Main Rd, Chennai 600004",
    "Af-684, Nandgram Near Mother India, Ghaziabad",
    "12 Rue de Rivoli, 75004 Paris",
    "",
    "null"
]

addr_results = []
for addr in address_samples:
    res = normalize_address(addr)
    addr_results.append({
        "Raw Address": addr if addr else "<EMPTY>",
        "Clean Norm": res["clean_norm"],
        "PIN Code": res["pin"],
        "Primary Number": res["primary_number"],
        "All Numbers": res["numbers"],
        "Is Missing?": res["is_missing"]
    })

display(pd.DataFrame(addr_results))

,Raw Address,Clean Norm,PIN Code,Primary Number,All Numbers,Is Missing?
0,"85 Wayne Avenue, Ticonderoga, NY 12883",85 wayne avenue ticonderoga ny 12883,12883,85,"[85, 12883]",False
1,"Wayne Ave, Ticonderoga Townshiip, NY",wayne avenueticonderoga townshiip ny,,,[],False
2,"6(29), C.I.T. Colony, 2nd Main Rd, Chennai 600004",6 29 c i t colony 2nd main roadchennai 600004,600004,6,"[6, 29, 600004]",False
3,"Af-684, Nandgram Near Mother India, Ghaziabad",af 684 nandgram near mother india ghaziabad,,684,[684],False
4,"12 Rue de Rivoli, 75004 Paris",12 rue de rivoli 75004 paris,75004,12,"[12, 75004]",False
5,<EMPTY>,,,,[],True
6,null,,,,[],True


## 5. Batch Transformation on Real Competition Dataset Sample
Demonstrating end-to-end transformation of competition records using `process_dataframe`.

In [6]:
print("Loading sample of real competition data...")
s1_train, s2_train, s3_train, _ = load_train_data(usecols=["entity_id", "business_name", "business_address", "country"])

sample_df = pd.concat([
    s1_train.head(5),
    s2_train.head(5),
    s3_train.head(5)
], ignore_index=True)

expanded_df = process_dataframe(sample_df)

display(expanded_df[[
    "entity_id", "name_clean", "name_core", "name_sorted",
    "addr_clean", "addr_pin", "addr_primary_num", "addr_is_missing", "country_norm"
]])

Loading sample of real competition data...


,entity_id,name_clean,name_core,name_sorted,addr_clean,addr_pin,addr_primary_num,addr_is_missing,country_norm
0,S1-925783039,orelee s barbershop,orelee s barbershop,barbershop orelee,1795 westchester drive high point nc,,1795,False,US
1,S1-773889195,prime money,prime money,money prime,17560 ellis road tahlequah ok,17560,17560,False,US
2,S1-377745466,b retail inc,b retail,retail,1712 montebello avenue phoenix az,,1712,False,US
3,S1-133037285,christ chapel,christ chapel,chapel christ,2100 cameron drive unit apartment g dundalk md,,2100,False,US
4,S1-755362802,prabhav business center,prabhav business center,business prabhav,797 lake town block a kolkata howrah west bengal,,797,False,INDIA
5,S2-166376419,र म म र क ट ग प र इव ट ल म ट ड,र म म र क ट ग प र इव ट ल म ट ड,इव,kh no 570 13 new delhi west delhi delhi,,570,False,INDIA
6,S2-764573417,holloway peak inc seafood,holloway peak seafood,holloway peak seafood,105 elm streetmorganton nc,,105,False,US
7,S2-639257739,आद त य प र पर ट ज एलएलप,आद त य प र पर ट ज एलएलप,आद एलएलप पर,g 3 571 gulmohar colony bhopal madhya pradesh,,3,False,INDIA
8,S2-163963287,summit inc,summit,summit,greensboro nc 19 1 2 stardust trail,,19,False,US
9,S2-49942811,delta tetlecommunication inc,delta tetlecommunication,delta tetlecommunication,914 pierpont avenuecleveland oh,,914,False,US


## 6. Summary & Readiness for Phase 3 (Blocking)

The Normalization Engine produces rich, orthogonal representations:
1. **`name_clean` & `name_core`**: Feed Exact and Core Name Hash Blocking.
2. **`name_sorted`**: Neutralizes word transposition variations.
3. **`name_informative_tokens`**: Directly populates the Inverted Index for Rare Token Blocking.
4. **`addr_pin` & `addr_primary_num`**: Form Composite Structural Address Anchors (`pin_name_prefix` / `num_name_prefix`).
5. **`addr_is_missing`**: Prevents False Negative penalties in downstream feature generation.

**Phase 2 Complete -> Ready for Phase 3: High-Recall Multi-Blocker & candidate_pairs.tsv Generation.**